# Day 34 — Kaggle Titanic Submission
### Feature Engineering · Predictions · Submission File · Leaderboard Score

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score
import pickle
import warnings

warnings.filterwarnings("ignore")

plt.style.use("dark_background")

# load both datasets
train = pd.read_csv(r"C:\DS-AI-75d\titanic.csv")
test = pd.read_csv(r"C:\DS-AI-75d\test.csv")

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"\nTest columns: {list(test.columns)}")
print(f"\nTest missing values:")
print(test.isnull().sum()[test.isnull().sum() > 0])
print(f"\nFirst 3 test passengers:")
print(test[["PassengerId", "Pclass", "Name", "Sex", "Age", "Fare"]].head(3))

Train: (891, 12)
Test:  (418, 11)

Test columns: ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Test missing values:
Age       86
Fare       1
Cabin    327
dtype: int64

First 3 test passengers:
   PassengerId  Pclass                              Name     Sex   Age    Fare
0          892       3                  Kelly, Mr. James    male  34.5  7.8292
1          893       3  Wilkes, Mrs. James (Ellen Needs)  female  47.0  7.0000
2          894       2         Myles, Mr. Thomas Francis    male  62.0  9.6875


## 2. Feature Engineering — Same Pipeline as Training

In [ ]:
print("=" * 55)
print("    FEATURE ENGINEERING — TRAIN + TEST")
print("=" * 55)


def engineer_features(df):
    df = df.copy()

    # fill missing values
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna("S")
    df["Cabin"] = df["Cabin"].fillna("Unknown")

    # family features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # fare transformation
    df["FareLog"] = np.log1p(df["Fare"])

    # cabin feature
    df["HasCabin"] = (df["Cabin"] != "Unknown").astype(int)

    # gender encoding
    df["Sex_encoded"] = (df["Sex"] == "female").astype(int)

    # title extraction
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    title_map = {"Mr": "Mr", "Miss": "Miss", "Mrs": "Mrs", "Master": "Master"}
    df["Title"] = df["Title"].map(title_map).fillna("Other")
    for t in ["Mr", "Mrs", "Miss", "Master"]:
        df[f"Title_{t}"] = (df["Title"] == t).astype(int)

    return df


train_eng = engineer_features(train)
test_eng = engineer_features(test)

features = [
    "Pclass",
    "Age",
    "FareLog",
    "FamilySize",
    "IsAlone",
    "HasCabin",
    "Sex_encoded",
    "Title_Mr",
    "Title_Mrs",
    "Title_Miss",
    "Title_Master",
]

X_train = train_eng[features]
y_train = train_eng["Survived"]
X_test = test_eng[features]

print(f"Train features: {X_train.shape}")
print(f"Test features:  {X_test.shape}")
print(f"\nFeatures: {features}")

print(f"\nTitle distribution in test set:")
print(test_eng["Title"].value_counts())

print(f"\nMissing values after engineering:")
print(f"  Train: {X_train.isnull().sum().sum()}")
print(f"  Test:  {X_test.isnull().sum().sum()}")
print("\nFeature engineering complete! ✅")

    FEATURE ENGINEERING — TRAIN + TEST
Train features: (891, 11)
Test features:  (418, 11)

Features: ['Pclass', 'Age', 'FareLog', 'FamilySize', 'IsAlone', 'HasCabin', 'Sex_encoded', 'Title_Mr', 'Title_Mrs', 'Title_Miss', 'Title_Master']

Title distribution in test set:
Title
Mr        240
Miss       78
Mrs        72
Master     21
Other       7
Name: count, dtype: int64

Missing values after engineering:
  Train: 0
  Test:  0

Feature engineering complete! ✅


## 3. Train Models & Generate Predictions

In [ ]:
print("=" * 55)
print("    TRAINING MODELS & GENERATING PREDICTIONS")
print("=" * 55)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# train all three models
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=4, random_state=42
    ),
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(random_state=42, max_iter=1000)),
        ]
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42
    ),
}

results = {}
for name, model in models.items():
    cv_auc = cross_val_score(model, X_train, y_train, cv=skf, scoring="roc_auc").mean()
    cv_acc = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy").mean()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        "model": model,
        "cv_auc": cv_auc,
        "cv_acc": cv_acc,
        "predictions": preds,
        "survived_count": preds.sum(),
    }
    print(f"\n{name}:")
    print(f"  CV AUC:      {cv_auc:.4f}")
    print(f"  CV Accuracy: {cv_acc:.4f}")
    print(f"  Predicted survivors: {preds.sum()}/418 ({preds.sum()/418*100:.1f}%)")

print(f"\n--- Prediction Agreement ---")
rf_preds = results["Random Forest"]["predictions"]
lr_preds = results["Logistic Regression"]["predictions"]
gb_preds = results["Gradient Boosting"]["predictions"]

agree = ((rf_preds == lr_preds) & (lr_preds == gb_preds)).sum()
print(f"All 3 models agree: {agree}/418 ({agree/418*100:.1f}%)")
print(f"RF vs LR disagree: {(rf_preds != lr_preds).sum()} passengers")
print(f"RF vs GB disagree: {(rf_preds != gb_preds).sum()} passengers")

# ensemble — majority vote
ensemble_preds = ((rf_preds + lr_preds + gb_preds) >= 2).astype(int)
print(f"\nEnsemble (majority vote):")
print(
    f"  Predicted survivors: {ensemble_preds.sum()}/418 ({ensemble_preds.sum()/418*100:.1f}%)"
)

    TRAINING MODELS & GENERATING PREDICTIONS

Random Forest:
  CV AUC:      0.8761
  CV Accuracy: 0.8384
  Predicted survivors: 159/418 (38.0%)

Logistic Regression:
  CV AUC:      0.8752
  CV Accuracy: 0.8227
  Predicted survivors: 168/418 (40.2%)

Gradient Boosting:
  CV AUC:      0.8713
  CV Accuracy: 0.8361
  Predicted survivors: 159/418 (38.0%)

--- Prediction Agreement ---
All 3 models agree: 363/418 (86.8%)
RF vs LR disagree: 31 passengers
RF vs GB disagree: 34 passengers

Ensemble (majority vote):
  Predicted survivors: 163/418 (39.0%)


## 4. Generate Kaggle Submission File

In [ ]:
print("=" * 55)
print("    GENERATING KAGGLE SUBMISSION FILES")
print("=" * 55)

# create submission files for each model + ensemble
submissions = {
    "rf_submission": results["Random Forest"]["predictions"],
    "lr_submission": results["Logistic Regression"]["predictions"],
    "gb_submission": results["Gradient Boosting"]["predictions"],
    "ensemble_submission": ensemble_preds,
}

for name, preds in submissions.items():
    sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": preds})
    filepath = rf"C:\DS-AI-75d\{name}.csv"
    sub.to_csv(filepath, index=False)
    print(f"✅ {name}.csv saved — {preds.sum()} survivors predicted")

# show the RF submission (our main model)
print(f"\n--- RF Submission Preview ---")
rf_sub = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": results["Random Forest"]["predictions"],
    }
)
print(rf_sub.head(10).to_string(index=False))

print(f"\n--- Ensemble Submission Preview ---")
ens_sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": ensemble_preds})
print(ens_sub.head(10).to_string(index=False))

print(f"""
WHICH FILE TO SUBMIT TO KAGGLE?
  1. rf_submission.csv     — our main model (CV AUC=0.876)
  2. ensemble_submission.csv — majority vote of all 3

Recommendation: submit BOTH and compare scores!
Kaggle allows up to 10 submissions per day.

HOW TO SUBMIT:
  1. Go to kaggle.com/competitions/titanic
  2. Click 'Submit Predictions'
  3. Upload rf_submission.csv
  4. Get your leaderboard score!
""")

    GENERATING KAGGLE SUBMISSION FILES
✅ rf_submission.csv saved — 159 survivors predicted
✅ lr_submission.csv saved — 168 survivors predicted
✅ gb_submission.csv saved — 159 survivors predicted
✅ ensemble_submission.csv saved — 163 survivors predicted

--- RF Submission Preview ---
 PassengerId  Survived
         892         0
         893         0
         894         0
         895         0
         896         1
         897         0
         898         1
         899         0
         900         1
         901         0

--- Ensemble Submission Preview ---
 PassengerId  Survived
         892         0
         893         0
         894         0
         895         0
         896         1
         897         0
         898         1
         899         0
         900         1
         901         0

WHICH FILE TO SUBMIT TO KAGGLE?
  1. rf_submission.csv     — our main model (CV AUC=0.876)
  2. ensemble_submission.csv — majority vote of all 3

Recommendation: submit BOT

## 5. Kaggle Results & Key Takeaways 🎯

### Submission Results
| Model | Predicted Survivors | Kaggle Score |
|---|---|---|
| Random Forest | 159/418 (38.0%) | **0.76794** |
| Logistic Regression | 168/418 (40.2%) | not submitted |
| Gradient Boosting | 159/418 (38.0%) | not submitted |
| Ensemble (majority vote) | 163/418 (39.0%) | **0.76794** |

### Score Context
- 0.76794 = correctly predicted 76.8% of test passengers
- Beats naive baseline (predict all female = 0.766)
- Top 25% of all submissions score above 0.77
- Top scores reach ~0.85 with heavy feature engineering

### Why CV AUC (0.876) ≠ Kaggle Score (0.768)?
- CV AUC measures ranking quality across ALL thresholds
- Kaggle score measures accuracy at threshold=0.5 only
- Different metric — not directly comparable
- Our train accuracy was 83.8% — Kaggle 76.8% shows some overfitting to train

### How to Improve the Kaggle Score
1. Better feature engineering (ticket prefix, deck from cabin)
2. Tune threshold (not just 0.5)
3. More aggressive title grouping
4. Stacking/blending multiple models
5. Use external data (historical Titanic records)

### Submission Files Created
- C:\DS-AI-75d\rf_submission.csv
- C:\DS-AI-75d\lr_submission.csv
- C:\DS-AI-75d\gb_submission.csv
- C:\DS-AI-75d\ensemble_submission.csv

### Key Lessons
- Always submit multiple models — they can differ
- CV score and Kaggle score use different metrics
- Ensemble didn't help here — models mostly agreed (86.8%)
- Feature engineering is the main lever for improvement